#Telecom Domain Read & Write Ops Hackathon - Building Datalake & Lakehouse
This notebook contains assignments to practice Spark read options and Databricks volumes. <br>
Sections: Sample data creation, Catalog & Volume creation, Copying data into Volumes, Path glob/recursive reads, toDF() column renaming variants, inferSchema/header/separator experiments, and exercises.<br>

![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

##First Import all required libraries & Create spark session object


## 1. Write SQL statements to create:


1. A catalog named telecom_catalog_assign
2. A schema landing_zone
3. A volume landing_vol

4. Using dbutils.fs.mkdirs, create folders:
- /Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
- /Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
- /Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/

5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):
a. Volume vs DBFS/FileStore
b. Why production teams prefer Volumes for regulated data


### 1.1 Import required libraries:

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Telecom_Read_Write_Hackathon") \
    .getOrCreate()

print("Spark Session Created Successfully")
print("Spark Version:", spark.version)


### 1.2 Create Catalog, Schema and Volume:

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS telecom_catalog_assign;

CREATE SCHEMA IF NOT EXISTS telecom_catalog_assign.landing_zone;

CREATE VOLUME IF NOT EXISTS telecom_catalog_assign.landing_zone.landing_vol;


### Set the current catalog and schema:

In [0]:
%sql

USE CATALOG telecom_catalog_assign;

USE SCHEMA landing_zone;


### 1.3 Create folders inside Volume:


4. Using **dbutils.fs.mkdirs**, create folders:
- /Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
- /Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
- /Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/

In [0]:
base_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"

dbutils.fs.mkdirs(f"{base_path}/customer")
dbutils.fs.mkdirs(f"{base_path}/usage")
dbutils.fs.mkdirs(f"{base_path}/tower/region1")
dbutils.fs.mkdirs(f"{base_path}/tower/region2")

print("All folders created successfully")


**5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):**

- a. Volume vs DBFS/FileStore.
- b. Why production teams prefer Volumes for regulated data.


A **Unity Catalog Volume** is a `governed storage location` used to `store` and `access` **non-tabular files** such as:

- CSV files
- JSON files
- XML files
- Parquet files
- Images and documents
- Raw source files
- Configuration files


- **DBFS (Databricks File System)** is a `distributed file-system interface`. 
- It is used to `users to access files` in **cloud storage** through Databricks.


### What is FileStore?


**FileStore** is a `legacy DBFS location` commonly used for `storing and sharing files`.


**It was commonly used for:**

- Uploading CSV files.
- Sharing files.
- Downloading files.
- Accessing files through browser URLs.


**Why production teams prefer Volumes:**

- **Centralized access control** through Unity Catalog.
- Better **governance and auditing**.
- Access can be controlled using **READ VOLUME, WRITE VOLUME** and **other privileges**.
- Suitable for managing **CSV, JSON, XML, images** and other **non-tabular files**.
- Supports a **governed data architecture**.


### Data files to use in this usecase:


### 1. Customer CSV Data:


```python
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''
```


### 2. Usage TSV Data:


```python
usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''
```



### 3. Tower Logs – Region 1 (Pipe-Delimited Data):


```python
tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''
```


## 2. Filesystem operations:


1. Write **dbutils.fs.put** code to copy the above datasets into your created Volume folders:

- Customer data → /Volumes/.../customer/
- Usage data → /Volumes/.../usage/
- Tower data (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/


**2. Write a command to validate whether files were successfully copied**


### 2.1 Prepare sample datasets:


Create the datasets exactly as multiline strings.


**Customer CSV:**

In [0]:
customer_csv = """101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID"""


**Usage TSV:**

In [0]:
usage_tsv = """customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0"""


**Tower logs – Region 1**

In [0]:
tower_logs_region1 = """event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12"""


### 2.2 Copy datasets into Volume using dbutils.fs.put()

In [0]:
dbutils.fs.put(
    f"{base_path}/customer/customer.csv",
    customer_csv,
    True
)

dbutils.fs.put(
    f"{base_path}/usage/usage.tsv",
    usage_tsv,
    True
)

dbutils.fs.put(
    f"{base_path}/tower/region1/tower_region1.csv",
    tower_logs_region1,
    True
)


### 2.3 Validate files:

In [0]:
display(dbutils.fs.ls(base_path))
display(dbutils.fs.ls(f"{base_path}/customer"))
display(dbutils.fs.ls(f"{base_path}/usage"))
display(dbutils.fs.ls(f"{base_path}/tower/region1"))


### To recursively inspect all files:

In [0]:
def list_files(path):
    for file in dbutils.fs.ls(path):
        if file.isDir():
            list_files(file.path)
        else:
            print(file.path, file.size, "bytes")

list_files(base_path)


**Interview explanation:** 

- **dbutils.fs.put()** is used to `creates or overwrites` a file at a `specified file system location`. 
- **dbutils.fs.ls()** is used to `lists` files and directories.


## 3. Spark Directory Read Use Cases:


**1. Read all tower logs using:**

- Path glob filter (example: *.csv)
- Multiple paths input
- Recursive lookup

**2. Demonstrate these 3 reads separately:**

- Using pathGlobFilter
- Using list of paths in spark.read.csv([path1, path2])
- Using .option("recursiveFileLookup","true")

**3. Compare the outputs and understand when each should be used.**


**We have tower logs in different regional folders.**


### First define the paths:

In [0]:
tower_path = f"{base_path}/tower/"

region1_path = f"{tower_path}/region1"
region2_path = f"{tower_path}/region2"

In [0]:
print(tower_path)

display(dbutils.fs.ls(tower_path))


### 3.1 Using pathGlobFilter:


**pathGlobFilter** is used to `filters files` based on their `filenames`.


**For example:**

- `*.csv` – All CSV files.
- `*.json` – All JSON files.
- `tower*.csv` – CSV files beginning with tower.

In [0]:
tower_glob_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", "|")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.csv")
        .load(tower_path)
)

display(tower_glob_df)


### 3.2 Using multiple paths:


**When region 1 data is available:**

In [0]:
tower_multiple_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", "|")
        .load([
            f"{region1_path}/tower_region1.csv"
        ])
)

display(tower_multiple_df)


**When region 1, 2 data is available:**

In [0]:
tower_multiple_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", "|")
        .load([
            f"{region1_path}/tower_region1.csv",
            f"{region2_path}/tower_region2.csv"
        ])
)


### 3.3 Using recursiveFileLookup:


**recursiveFileLookup** is used to `searches files` inside **nested sub-directories**.

In [0]:
tower_recursive_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", "|")
        .option("recursiveFileLookup", "true")
        .load(tower_path)
)

display(tower_recursive_df)


## 4. Schema Inference, Header, and Separator:


**1. Try the Customer, Usage files with the option and options using `read.csv` and `format function`:**

- header=false, 
- inferSchema=false

or

- header=true, 
- inferSchema=true

**2. Write a note on What changed when we use `header or inferSchema`  with true/false?**

**3. How schema inference handled “abc” in age?**


### 4.1 Customer CSV – header=false, inferSchema=false:

In [0]:
customer_df1 = (
    spark.read
        .format("csv")
        .option("header", "false")
        .option("inferSchema", "false")
        .option("sep", ",")
        .load(f"{base_path}/customer/customer.csv")
)

display(customer_df1)
customer_df1.printSchema()


**Why?**

- `header=false:` First row is treated as data.
- `inferSchema=false:` All columns are initially read as strings.
- Spark automatically assigns _c0, _c1, etc.


### 4.2 Customer CSV – header=true, inferSchema=true:

In [0]:
customer_df2 = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ",")
        .load(f"{base_path}/customer/customer.csv")
)

display(customer_df2)
customer_df2.printSchema()


### 4.3 Correct customer read:

In [0]:
customer_df = (
    spark.read
        .format("csv")
        .option("header", "false")
        .option("inferSchema", "true")
        .option("sep", ",")
        .load(f"{base_path}/customer/customer.csv")
        .toDF(
            "customer_id",
            "customer_name",
            "age",
            "city",
            "plan_type"
        )
)

display(customer_df)
customer_df.printSchema()


### 4.4 Usage TSV – header and separator:

In [0]:
usage_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", "\t")
        .load(f"{base_path}/usage/usage.tsv")
)

display(usage_df)
usage_df.printSchema()


### 4.5 Header vs inferSchema

| Option | False | True |
|---|---|---|
| `header` | First row is treated as data | First row contains column names |
| `inferSchema` | Columns are initially treated as StringType | Spark automatically determines compatible data types |



## 5. Column Renaming Usecases:


1. Apply **column names** using string using `toDF` function for customer data
2. Apply **column names and datatype** using the `schema` function for usage data
3. Apply **column names and datatype** using the `StructType` with `IntegerType`, `StringType`, `TimestampType` and other classes for towers data 


### 5.1 Rename columns using toDF():


**toDF()** is used to `rename all DataFrame columns` at once.

In [0]:
customer_renamed_df = customer_df1.toDF(
    "customer_id",
    "customer_name",
    "age",
    "city",
    "plan_type"
)

display(customer_renamed_df)


### 5.2 Apply column names and data types using schema():

In [0]:
usage_schema = """
customer_id INT,
voice_mins INT,
data_mb INT,
sms_count INT
"""

usage_schema_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", "\t")
        .schema(usage_schema)
        .load(f"{base_path}/usage/usage.tsv")
)

display(usage_schema_df)
usage_schema_df.printSchema()


### 5.3 Tower data using StructType:

In [0]:
tower_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("tower_id", StringType(), True),
    StructField("signal_strength", IntegerType(), True),
    StructField("timestamp", TimestampType(), True)
])


### Read the tower file:

In [0]:
tower_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", "|")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .schema(tower_schema)
        .load(f"{region1_path}/tower_region1.csv")
)

display(tower_df)
tower_df.printSchema()


**Interview explanation:**

- **StructType** defines the `complete DataFrame schema`.
- **StructField** defines an `individual column`.
- **IntegerType, StringType, TimestampType** specify `column datatypes`.
- **nullable=True** allows `null values`.

## Spark Write Operations using 
- csv, json, orc, parquet, delta, saveAsTable, insertInto, xml with different write mode, header and sep options


### 5.4 Spark Write Format Overview

| Format | Write Syntax |
|---|---|
| CSV | `.write.csv(path)` |
| JSON | `.write.json(path)` |
| ORC | `.write.orc(path)` |
| Parquet | `.write.parquet(path)` |
| Delta | `.write.format("delta").save(path)` |
| Table | `.write.saveAsTable("table_name")` |
| Insert | `.write.insertInto("table_name")` |
| XML | `.write.format("xml").save(path)` |


In [0]:
.mode("overwrite")
.mode("append")
.mode("ignore")
.mode("error")


##6. Write Operations (Data Conversion/Schema migration) – CSV Format Usecases:


1. Write **customer data into CSV format** using `overwrite mode`
2. Write **usage data into CSV format** using `append mode`
3. Write **tower data into CSV format** with `header enabled` and `custom separator (|)`
4. Read the **tower data in a dataframe** and `show only 5 rows`.
5. Download the file into local from the catalog volume location and see the data of any of the above files opening in a notepad++.


**We will write each DataFrame into a separate output directory:**

In [0]:
output_path = f"{base_path}/output"

dbutils.fs.mkdirs(output_path)


### 6.1 Customer data – overwrite mode:

In [0]:
customer_df1.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .save(f"{output_path}/csv/customer")


### 6.2 Usage data – append mode:

In [0]:
usage_df.write \
    .format("csv") \
    .mode("append") \
    .option("header", "true") \
    .option("sep", "\t") \
    .save(f"{output_path}/csv/usage")


### 6.3 Tower data – custom separator:

In [0]:
tower_df.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", "|") \
    .save(f"{output_path}/csv/tower")


### 6.4 Read tower CSV and display five rows:

In [0]:
tower_csv_read = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", "|")
    .schema(tower_schema)
    .load(f"{output_path}/csv/tower")
)

display(tower_csv_read.limit(5))


### 6.5 Inspect generated files:

In [0]:
display(dbutils.fs.ls(f"{output_path}/csv/tower"))


**To download a file,** 

- use the Databricks Catalog Explorer to navigate to the Volume, 
- download the generated part file, if your workspace supports it.


## 7. Write Operations (Data Conversion/Schema migration)– JSON Format Usecases:


1. Write **customer data into JSON format** using `overwrite mode`
2. Write **usage data into JSON format** using a`ppend mode and snappy compression format`
3. Write **tower data into JSON format** using `ignore mode and observe the behavior of this mode`
4. Read the **tower data in a dataframe** and `show only 5 rows`.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.


### 7.1 Customer – overwrite mode:

In [0]:
customer_df1.write \
    .format("json") \
    .mode("overwrite") \
    .save(f"{output_path}/json/customer")


### 7.2 Usage – append mode with Snappy compression:


**Snappy compression** is used to reduces `storage consumption` and commonly used in `analytical data processing`.

In [0]:
usage_df.write \
    .format("json") \
    .mode("append") \
    .option("compression", "snappy") \
    .save(f"{output_path}/json/usage")


### 7.3 Tower – ignore mode:


- **Ignore mode** is a `write mode in Spark` used to **skips the write operation** if the destination already exists. 
- It does not throw an **error** and does not **overwrite existing data**.

In [0]:
tower_df.write \
    .format("json") \
    .mode("ignore") \
    .save(f"{output_path}/json/tower")


### Test it:

In [0]:
display(dbutils.fs.ls(f"{output_path}/json/tower"))


### 7.4 Read JSON tower data:

In [0]:
tower_json_read = spark.read \
    .format("json") \
    .load(f"{output_path}/json/tower")

display(tower_json_read.limit(5))


## 8. Write Operations (Data Conversion/Schema migration) – Parquet Format Usecases


1. Write `customer data into Parquet format` using **overwrite mode** and in a **gzip format**
2. Write `usage data into Parquet format` using **error mode**
3. Write `tower data into Parquet format` with **gzip compression option**
4. Read the `usage data in a dataframe` and `show only 5 rows`.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.


- **Parquet** is a `binary columnar format`. 
- Opening a **Parquet file** in Notepad++ will not show readable tabular data. 
- Use **Spark or a Parquet viewer** to inspect its contents.


### 8.1 Customer – overwrite with GZIP:

In [0]:
customer_df1.write \
    .format("parquet") \
    .mode("overwrite") \
    .option("compression", "gzip") \
    .save(f"{output_path}/parquet/customer")


### 8.2 Usage – error mode:

In [0]:
usage_df.write \
    .format("parquet") \
    .mode("error") \
    .save(f"{output_path}/parquet/usage")


**error** is the `default write mode`.


### 8.3 Tower – GZIP compression:

In [0]:
tower_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .option("compression", "gzip") \
    .save(f"{output_path}/parquet/tower")


### 8.4 Read usage Parquet:

In [0]:
usage_parquet_read = spark.read \
    .format("parquet") \
    .load(f"{output_path}/parquet/usage")

display(usage_parquet_read.limit(5))
usage_parquet_read.printSchema()


## 9. Write Operations (Data Conversion/Schema migration) – Orc Format Usecases:


1. Write **customer data into ORC format** using `overwrite mode`
2. Write **usage data into ORC format** using `append mode`
3. Write **tower data into ORC format** and see the `output file structure`
4. Read the **usage data in a dataframe** and `show only 5 rows`.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.


- **ORC** stands for `Optimized Row Columnar`.
- It is a `columnar storage format` used for `analytical workloads`.


### 9.1 Customer – overwrite:

In [0]:
customer_df1.write \
    .format("orc") \
    .mode("overwrite") \
    .save(f"{output_path}/orc/customer")


### 9.2 Usage – append:

In [0]:
usage_df.write \
    .format("orc") \
    .mode("append") \
    .save(f"{output_path}/orc/usage")


### 9.3 Tower – ORC:

In [0]:
tower_df.write \
    .format("orc") \
    .mode("overwrite") \
    .save(f"{output_path}/orc/tower")


### 9.4 Inspect ORC files:

In [0]:
display(dbutils.fs.ls(f"{output_path}/orc/tower"))


### 9.5 Read usage ORC:

In [0]:
usage_orc_read = spark.read \
    .format("orc") \
    .load(f"{output_path}/orc/usage")

display(usage_orc_read.limit(5))


## 10. Write Operations (Data Conversion/Schema migration) – Delta Format Usecases:


1. Write **customer data into Delta format** using `overwrite mode`
2. Write **usage data into Delta format** using `append mode`
3. Write **tower data into Delta format** and `see the output file structure`
4. Read the **usage data in a dataframe** and `show only 5 rows`.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.
6. Compare the **parquet location and delta location** and try to understand what is the differentiating factor, as both are parquet files only.


### 10.1 Customer – overwrite:

In [0]:
customer_df1.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{output_path}/delta/customer")


### 10.2 Usage – append:

In [0]:
usage_df.write \
    .format("delta") \
    .mode("append") \
    .save(f"{output_path}/delta/usage")


### 10.3 Tower – Delta:

In [0]:
tower_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{output_path}/delta/tower")


### 10.4 Inspect Delta directory:

In [0]:
display(dbutils.fs.ls(f"{output_path}/delta/tower"))


### 10.5 Read Delta usage:

In [0]:
usage_delta_read = spark.read \
    .format("delta") \
    .load(f"{output_path}/delta/usage")

display(usage_delta_read.limit(5))


**Interview answer:**

- **Delta Lake** uses `Parquet files` for **physical data storage** and **maintains a transaction log** in _delta_log. 
- It provides `ACID transactions, schema enforcement, time travel` and `reliable concurrent operations`.


## 11. Write Operations (Lakehouse Usecases) – Delta table Usecases:


1. Write **customer data** using `saveAsTable()` as a managed table
2. Write **usage data** using `saveAsTable()` with overwrite mode
3. Drop the **managed table** and verify `data removal`
4. Go and check the **table overview** and realize it is in `delta format in the Catalog`.
5. Use **spark.read.sql** to `write some simple queries` on the above tables created.



### 11.1 Customer – managed table:

In [0]:
customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "telecom_catalog_assign.landing_zone.customer_managed"
    )


### 11.2 Usage – managed table with overwrite:

In [0]:
usage_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "telecom_catalog_assign.landing_zone.usage_managed"
    )


### 11.3 Verify tables:

In [0]:
%sql

SHOW TABLES IN telecom_catalog_assign.landing_zone;


### Read the tables:

In [0]:
customer_table_df = spark.read.table(
    "telecom_catalog_assign.landing_zone.customer_managed"
)

usage_table_df = spark.read.table(
    "telecom_catalog_assign.landing_zone.usage_managed"
)

display(customer_table_df)
display(usage_table_df)


### 11.4 Execute SQL queries:

In [0]:
spark.sql("""
SELECT *
FROM telecom_catalog_assign.landing_zone.customer_managed
""").show()


### Find customer count by plan:

In [0]:
spark.sql("""
SELECT plan_type, COUNT(*) AS customer_count
FROM telecom_catalog_assign.landing_zone.customer_managed
GROUP BY plan_type
""").show()


### Find total usage:

In [0]:
spark.sql("""
SELECT
    SUM(voice_mins) AS total_voice_mins,
    SUM(data_mb) AS total_data_mb,
    SUM(sms_count) AS total_sms
FROM telecom_catalog_assign.landing_zone.usage_managed
""").show()


### 11.5 Drop a managed table:


**First create a separate test table:**

In [0]:
customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "telecom_catalog_assign.landing_zone.customer_drop_test"
    )


**Drop it:**

In [0]:
%sql
DROP TABLE telecom_catalog_assign.landing_zone.customer_drop_test;


**Verify:**

In [0]:
%sql
SHOW TABLES IN telecom_catalog_assign.landing_zone;


### 12. Write Operations (Lakehouse Usecases) – Delta table Usecases:


1. Write **customer data using insertInto() in a new table** and find the behav`ior
2. Write **usage data using insertTable()** with `overwrite mode`


- **insertInto()** is used to `inserts a records into an existing table`.
- It `does not create a table `automatically.


### 12.1 Customer – create destination table:

In [0]:
customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "telecom_catalog_assign.landing_zone.customer_insert"
    )


**For a clean insert demonstration, empty the destination first:**

In [0]:
spark.sql("""
TRUNCATE TABLE telecom_catalog_assign.landing_zone.customer_insert
""")


### Insert customer records:

In [0]:
customer_df.write \
    .mode("append") \
    .insertInto(
        "telecom_catalog_assign.landing_zone.customer_insert"
    )


**Verify:**

In [0]:
%sql

SELECT *
FROM telecom_catalog_assign.landing_zone.customer_insert;


### 12.2 Usage – insertInto with overwrite:


**Create the destination:**

In [0]:
usage_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "telecom_catalog_assign.landing_zone.usage_insert"
    )


**Overwrite the table using insertInto:**

In [0]:
usage_df.write \
    .mode("overwrite") \
    .insertInto(
        "telecom_catalog_assign.landing_zone.usage_insert"
    )


**Verify:**

In [0]:
%sql

SELECT *
FROM telecom_catalog_assign.landing_zone.usage_insert;


**Important interview points:**

- **saveAsTable()** can create a new table.
- **insertInto()** requires an existing table.
- **insertInto()** matches columns by position, not by column name.
- Ensure the source and destination column order and datatypes match.
- **mode("overwrite")** replaces existing table data.

Also, 

- **insertTable()** is not the standard PySpark DataFrameWriter method. 
- Use **insertInto()** or **SQL INSERT INTO**.


### 13. Write Operations (Lakehouse Usecases) – Delta table Usecases:


1. Write **customer data into XML format** using **rowTag as cust**
2. Write **usage data into XML format** using `overwrite mode` with the rowTag as usage
3. Download the xml data and open the file in notepad++ and see how the xml file looks like.


**XML** is a `text-based hierarchical data format`.


### 13.1 Customer – XML using rowTag:

In [0]:
customer_df.write \
    .format("xml") \
    .mode("overwrite") \
    .option("rowTag", "cust") \
    .save(f"{output_path}/xml/customer")


**rowTag** is a `XML element` that Spark should treat as `one individual row (record)` when **reading XML data**.


### 13.2 Usage – XML using overwrite:

In [0]:
usage_df.write \
    .format("xml") \
    .mode("overwrite") \
    .option("rowTag", "usage") \
    .save(f"{output_path}/xml/usage")


### 13.3 Read XML:

In [0]:
customer_xml_read = (
    spark.read
    .format("xml")
    .option("rowTag", "cust")
    .load(f"{output_path}/xml/customer")
)

display(customer_xml_read)


**Read usage:**

In [0]:
usage_xml_read = (
    spark.read
        .format("xml")
        .option("rowTag", "usage")
        .load(f"{output_path}/xml/usage")
)

display(usage_xml_read)


### 13.4 Understand XML output:


```text
<cust>
    <customer_id>101</customer_id>
    <customer_name>Arun</customer_name>
    <age>31</age>
    <city>Chennai</city>
    <plan_type>PREPAID</plan_type>
</cust>
```


### 14. Compare all the downloaded files (csv, json, orc, parquet, delta and xml):


1. Capture the size occupied between all of these file formats and list the formats below based on the order of size from small to big.


### 14.1 Calculate directory size recursively:

In [0]:
def get_directory_size(path):
    total_size = 0

    for file in dbutils.fs.ls(path):
        if file.isDir():
            total_size += get_directory_size(file.path)
        else:
            if not file.name.startswith("_"):
                total_size += file.size

    return total_size


### 14.2 Compare customer data sizes:

In [0]:
formats = [
    "csv",
    "json",
    "parquet",
    "orc",
    "delta",
    "xml"
]

size_results = []

for fmt in formats:
    path = f"{output_path}/{fmt}/customer"

    size_bytes = get_directory_size(path)

    size_results.append((fmt, size_bytes))

size_df = spark.createDataFrame(
    size_results,
    ["file_format", "size_bytes"]
)

display(size_df.orderBy("size_bytes"))


**Convert bytes into KB:**

In [0]:
size_df_kb = size_df.withColumn(
    "size_kb",
    round(col("size_bytes") / 1024, 2)
)

display(size_df_kb.orderBy("size_bytes"))


### 14.3 Compare Delta transaction log size:

In [0]:
delta_log_size = get_directory_size(
    f"{output_path}/delta/customer/_delta_log"
)

print("Delta transaction log size:", delta_log_size, "bytes")


### 14.4 Expected observations:


**Typically:**

- JSON can be larger because of repeated field names.
- CSV can be compact for simple records but does not store schema information.
- Parquet and ORC are usually efficient for analytical workloads.
- Delta data files are Parquet files, with additional transaction-log storage.
- XML can be larger due to opening and closing tags.


### 15. Try to do permutation and combination of performing Schema Migration & Data Conversion operations like...


1. Read any one of the above **orc data in a dataframe** and `write it to dbfs in a parquet format`
2. Read any one of the above **parquet data in a dataframe** and `write it to dbfs in a delta format`
3. Read any one of the above **delta data in a dataframe** and `write it to dbfs in a xml format`
4. Read any one of the above **delta table in a dataframe** and `write it to dbfs in a json format`
5. Read any one of the above **delta table in a dataframe** and `write it to another table`


### 15.1 ORC to Parquet:

In [0]:
orc_df = spark.read \
    .format("orc") \
    .load(f"{output_path}/orc/customer")

orc_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(f"{output_path}/conversion/orc_to_parquet")


### 15.2 Parquet to Delta:

In [0]:
parquet_df = spark.read \
    .format("parquet") \
    .load(f"{output_path}/parquet/customer")

parquet_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{output_path}/conversion/parquet_to_delta")


**Verify:**

In [0]:
converted_delta_df = spark.read \
    .format("delta") \
    .load(f"{output_path}/conversion/parquet_to_delta")

display(converted_delta_df)


### 15.3 Delta to XML:

In [0]:
delta_df = spark.read \
    .format("delta") \
    .load(f"{output_path}/delta/customer")

delta_df.write \
    .format("xml") \
    .mode("overwrite") \
    .option("rowTag", "customer") \
    .save(f"{output_path}/conversion/delta_to_xml")


### 15.4 Delta to JSON:

In [0]:
delta_df = spark.read \
    .format("delta") \
    .load(f"{output_path}/delta/customer")

delta_df.write \
    .format("json") \
    .mode("overwrite") \
    .save(f"{output_path}/conversion/delta_to_json")


### 15.5 Delta table to another table:


**Read an existing Delta table:**

In [0]:
source_df = spark.read.table(
    "telecom_catalog_assign.landing_zone.customer_managed"
)


**Write into a new table:**

In [0]:
source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "telecom_catalog_assign.landing_zone.customer_copy"
    )


**Verify:**

In [0]:
%sql

SELECT *
FROM telecom_catalog_assign.landing_zone.customer_copy;


### 15.6 DBFS vs Unity Catalog Volume:


Your assignment mentions DBFS.


**Legacy DBFS destination example:**

In [0]:
dbfs_path = "dbfs:/FileStore/telecom/customer"

orc_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(dbfs_path)

In [0]:
volume_path = f"{base_path}/output/conversion/orc_to_parquet"

orc_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(volume_path)


## 16. Do a final exercise of defining one/two liner of... 


1. When to **use/benifits** `csv`
2. When to **use/benifits** `json`
3. When to **use/benifit **`orc`
4. When to **use/benifit** `parquet`
5. When to **use/benifit** `delta`
6. When to **use/benifit** `xml`
7. When to **use/benifit delta** `tables`



## 1. File Formats – Must-Remember Points:


| File Format | Key Interview Point |
|---|---|
| **CSV** | Simple, row-based, human-readable tabular format. |
| **JSON** | Supports nested and semi-structured data; commonly used in APIs. |
| **ORC** | Columnar format optimized for analytical queries, compression and Hive workloads. |
| **Parquet** | Columnar format widely used in Spark and data lakes. |
| **Delta Lake** | Adds ACID transactions, schema enforcement, time travel and MERGE support on top of Parquet files. |
| **XML** | Hierarchical format commonly used in legacy systems and enterprise integrations. |


## 2. Most Important Differences:


| Comparison | Key Difference |
|---|---|
| CSV vs JSON | Tabular data vs nested/hierarchical data |
| ORC vs Parquet | Both are columnar formats and support compression and efficient analytical reads. |
| Parquet vs Delta | Parquet is a file format; Delta Lake adds transaction management and table features. |
| Delta Lake vs Delta Table | Delta Lake is the storage technology; a Delta table is a table created using it. |
| Structured vs Semi-structured | Fixed tabular schema vs flexible or nested structure. |


## 3. Important Technical Terms:


### 3.1 Columnar Storage
- Stores data by column rather than by row.
- Improves analytical query performance when only selected columns are required.
- Example: Parquet and ORC.

### 3.2 Compression
- Reduces storage consumption.
- Can reduce data I/O and improve read performance.

### 3.3 Schema Enforcement
- Prevents incompatible data from being written to a Delta table.
- Helps maintain data consistency.

### 3.4 Schema Evolution
- Allows supported schema changes, such as adding new columns.
- Useful when source data structures change over time.

### 3.5 ACID Transactions
- Provides reliable transaction behavior.
- ACID stands for:
  - Atomicity
  - Consistency
  - Isolation
  - Durability

### 3.6 Time Travel
- Allows querying previous versions of a Delta table.
- Useful for auditing, debugging and recovering historical data.

### 3.7 MERGE
- Supports INSERT, UPDATE and conditional DELETE operations based on matching conditions.
- Commonly used for incremental data loading and upsert operations.

### 3.8 Predicate Pushdown
- Pushes filter conditions closer to the data source.
- Helps reduce unnecessary data reads.

### 3.9 Column Pruning
- Reads only the required columns instead of the entire dataset.
- Reduces unnecessary data scanning.


## 4. Real-Time Project Examples:


| Scenario | Suitable Format |
|---|---|
| Customer details received from an external source | CSV |
| Nested order information received through an API | JSON |
| Large-scale analytical processing | Parquet / ORC |
| Incremental customer updates | Delta |
| Historical transaction data from a legacy system | XML |
| Bronze, Silver and Gold architecture | Delta Tables |



## 5. Top 5 Interview Questions:


**High Priority – Prepare These Thoroughly**

1. What is the difference between Parquet and Delta?
2. Why do we use Delta Lake instead of Parquet in production?
3. What is the difference between ORC and Parquet?
4. What are the advantages of columnar storage?
5. What are ACID transactions in Delta Lake?


## 6. Interview Preparation Priority:


For a Data Engineer interview targeting 1.5–2 years of experience, prioritize these topics:

1. **Delta Lake and Parquet**
2. **ACID Transactions**
3. **Schema Enforcement and Schema Evolution**
4. **MERGE – INSERT, UPDATE and DELETE**
5. **Time Travel**
6. **Columnar Storage and Compression**
7. **Real-Time Pipeline Use Cases**


### Important Interview Tip:


Do not memorize only definitions. Be prepared to explain:
- What the format is.
- Why we use it.
- Its advantages.
- How it works in a real-time project.
- Why you selected it over another format.


### Remember: 


Delta Lake, Parquet and their real-time applications deserve more preparation time than the basic definitions of CSV, JSON and XML.